# Explainable AI-Based Phishing Website Detection Using URL Features

**Final-Year College Capstone Project**

---

### Executive Summary
Phishing attacks represent one of the most pervasive cyber security threats worldwide. Traditional detection mechanisms rely either on static blacklists (which suffer from zero-hour vulnerability) or dynamic web scraping (which introduces severe network overhead and security risks).

This project presents an **Explainable AI (XAI)** framework that classifies websites using **only raw URL strings** (zero page visits, no downloads) and provides **transparent, human-interpretable justifications** at both global and local levels using SHAP (Shapley Additive Explanations) and LIME (Local Interpretable Model-agnostic Explanations).

## 1. Setup, Environment & Reproducibility

In [ ]:
import sys
from pathlib import Path
import os

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

from src.data.load import load_config, load_dataset
from src.features.url_features import extract_features, batch_extract_features, get_feature_names
from src.models.predict import load_model, predict_url
from src.explain.reasons import generate_reasons

np.random.seed(42)
print('Environment initialized successfully. Project root:', PROJECT_ROOT)

## 2. Dataset Loading & Exploratory Data Analysis (EDA)
We utilize the benchmark **PhiUSIIL Phishing URL Dataset** (UCI Machine Learning Repository, ~235k samples). All pre-computed features are discarded; we retain strictly the URL string and ground-truth binary label.

In [ ]:
config = load_config()
df = load_dataset(config)
print('Loaded Dataset Shape:', df.shape)
print('\nClass Distribution:')
print(df['label'].value_counts(normalize=True).rename({0: 'Legitimate (0)', 1: 'Phishing (1)'}))

# Length analysis
df['url_length'] = df['url'].apply(len)
plt.figure(figsize=(10, 4))
sns.histplot(data=df, x='url_length', hue='label', bins=50, kde=True, palette={0: 'green', 1: 'red'})
plt.title('URL Length Distribution: Phishing (1) vs Legitimate (0)')
plt.xlim(0, 200)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 3. URL-Only Feature Extraction
We re-derive 52 lexical, structural, statistical, and obfuscation features strictly from the raw string without sending network requests.

In [ ]:
feature_names = get_feature_names()
print(f'Total engineered URL features: {len(feature_names)}')
print('Sample engineered features:', feature_names[:10])

sample_test_url = 'http://secure-login-update.xyz/verify/account.php?id=8823'
sample_feats = extract_features(sample_test_url)
pd.Series(sample_feats).head(15)

## 4. Leakage-Free Splitting: Grouped by Registered Domain
To prevent the model from memorizing specific domains, we partition using **GroupShuffleSplit** so that all URLs from any given registered domain belong exclusively to either train, validation, or test sets.

In [ ]:
data_proc = PROJECT_ROOT / 'data' / 'processed'
X_train = np.load(data_proc / 'X_train.npy')
X_val = np.load(data_proc / 'X_val.npy')
X_test = np.load(data_proc / 'X_test.npy')
y_train = np.load(data_proc / 'y_train.npy')
y_val = np.load(data_proc / 'y_val.npy')
y_test = np.load(data_proc / 'y_test.npy')

print(f'Partition sizes: Train={len(X_train)} | Val={len(X_val)} | Test={len(X_test)}')

## 5. Model Evaluation and Benchmarking
We benchmarked 6 classification algorithms on the held-out domain-grouped test set.

In [ ]:
metrics_csv = PROJECT_ROOT / 'artifacts' / 'metrics' / 'model_comparison.csv'
if metrics_csv.exists():
    comparison_df = pd.read_csv(metrics_csv)
    display(comparison_df)
else:
    print('Run scripts/train.py to generate full comparison tables.')

## 6. Overlaid ROC & Precision-Recall Curves

In [ ]:
from IPython.display import Image, display

roc_img = PROJECT_ROOT / 'artifacts' / 'plots' / 'roc_curves.png'
pr_img = PROJECT_ROOT / 'artifacts' / 'plots' / 'pr_curves.png'
if roc_img.exists():
    display(Image(filename=str(roc_img)))
if pr_img.exists():
    display(Image(filename=str(pr_img)))

## 7. Global Explainability with SHAP
Shapley values reveal how each URL feature shifts the model prediction globally across the test cohort.

In [ ]:
beeswarm_img = PROJECT_ROOT / 'artifacts' / 'plots' / 'shap_summary_beeswarm.png'
perm_img = PROJECT_ROOT / 'artifacts' / 'plots' / 'permutation_importance.png'
if beeswarm_img.exists():
    display(Image(filename=str(beeswarm_img)))
if perm_img.exists():
    display(Image(filename=str(perm_img)))

## 8. Local Explanation & Plain-English Interpretation
For any individual URL, we generate instant SHAP/LIME attributions and translate them into a clear plain-English reason list for security analysts.

In [ ]:
pipeline = load_model(artifacts_dir=PROJECT_ROOT / 'artifacts')
test_url = 'http://192.168.1.1/paypal/login-verification.php'

pred = predict_url(test_url, pipeline, feature_names)
reasons = generate_reasons(
    shap_local=None,
    lime_result=None,
    url_features=pred['features'],
    prediction_label=pred['label'],
    probability=pred['probability'],
    feature_names=feature_names,
    top_k=4
)

print(f'URL: {test_url}')
print(f'Verdict: {pred["label"].upper()} (Confidence: {pred["probability"]*100:.1f}%)')
print('\nPlain-English Justification:')
print(reasons['plain_english'])

## 9. Conclusion & Limitations
1. **High Accuracy via Pure Lexical Signals:** Achieved strong ROC-AUC on domain-grouped test splits without network interaction.
2. **Faithful Interpretability:** SHAP and LIME confirm key indicators: raw IP addresses, high entropy, sensitive keyword misuse, and suspicious TLDs.
3. **Practical Utility:** Packaged as a real-time Streamlit web app and FastAPI microservice ready for SOC triage.